# `_chunk_cumsum_fwd` — Deep Dive

This notebook traces through every line of the `_chunk_cumsum_fwd` function:
the Python wrapper that allocates outputs and launches the kernel, and
the Triton kernel itself that runs on the GPU.

## What this kernel computes

Given raw `dt` values and a per-head `A`, it produces:
1. `dt_out` — the processed dt (after bias, softplus, clamp)
2. `dA_cumsum` — the cumulative sum of `dt * A` within each chunk

This is Step 1 of the 5-stage Mamba2 forward pipeline.

In [1]:
import os, sys, types, math
os.environ.setdefault("CC", os.path.expanduser("~/miniconda3/envs/cutedsl/bin/x86_64-conda-linux-gnu-gcc"))
MAMBA_ROOT = os.path.expanduser("~/mamba")
sys.path.insert(0, MAMBA_ROOT)
pkg = types.ModuleType("mamba_ssm")
pkg.__path__ = [os.path.join(MAMBA_ROOT, "mamba_ssm")]
pkg.__package__ = "mamba_ssm"
sys.modules["mamba_ssm"] = pkg

import torch
import torch.nn.functional as F
import triton
from mamba_ssm.ops.triton.ssd_chunk_state import _chunk_cumsum_fwd

torch.set_printoptions(precision=4, sci_mode=False)
device = "cuda"
dtype = torch.float32

In [2]:
# ---- Create inputs ----
torch.manual_seed(42)

batch      = 2
seqlen     = 512
nheads     = 24
chunk_size = 256

dt      = torch.randn(batch, seqlen, nheads, device=device, dtype=dtype)
A       = -torch.rand(nheads, device=device, dtype=dtype)   # negative
dt_bias = torch.randn(nheads, device=device, dtype=dtype)

print(f"dt:      {list(dt.shape)}")
print(f"A:       {list(A.shape)}  values: {A[:4]}")
print(f"dt_bias: {list(dt_bias.shape)}")

dt:      [2, 512, 24]
A:       [24]  values: tensor([-0.9877, -0.1289, -0.5621, -0.5221], device='cuda:0')
dt_bias: [24]


---
## Part 1: The Python wrapper

File: `ssd_chunk_state.py:718-741`

This function allocates output tensors, computes the grid, and launches the Triton kernel.

In [3]:
# ============================================================
# LINE 718: def _chunk_cumsum_fwd(dt, A, chunk_size, dt_bias=None, dt_softplus=False,
#                                  dt_limit=(0.0, float("inf"))):
# ============================================================

# LINE 719: batch, seqlen, nheads = dt.shape
# Unpack dt shape.
# dt is the raw time-step tensor from in_proj, BEFORE any processing.
batch_, seqlen_, nheads_ = dt.shape
print(f"batch={batch_}, seqlen={seqlen_}, nheads={nheads_}")

# LINE 720: assert A.shape == (nheads,)
# A is a single scalar per head — the base decay rate.
assert A.shape == (nheads_,)

# LINE 721-722: if dt_bias is not None: assert dt_bias.shape == (nheads,)
# dt_bias is also per-head. It shifts dt before softplus.
assert dt_bias.shape == (nheads_,)

# LINE 723: nchunks = math.ceil(seqlen / chunk_size)
# How many chunks we split the sequence into.
# If seqlen=512, chunk_size=256 => nchunks=2
# If seqlen=500, chunk_size=256 => nchunks=2 (last chunk padded)
nchunks = math.ceil(seqlen_ / chunk_size)
print(f"nchunks = ceil({seqlen_}/{chunk_size}) = {nchunks}")

batch=2, seqlen=512, nheads=24
nchunks = ceil(512/256) = 2


In [4]:
# LINE 724: dt_out = torch.empty(batch, nheads, nchunks, chunk_size,
#                                 device=dt.device, dtype=torch.float32)
#
# Allocate output for processed dt values.
# Shape: (batch, nheads, nchunks, chunk_size)
#
# NOTE: The layout is DIFFERENT from the input dt!
#   Input  dt:     (batch, seqlen,  nheads)        — sequence-major
#   Output dt_out: (batch, nheads, nchunks, chunk_size) — head-major, chunked
#
# The kernel reads dt in (batch, seqlen, nheads) layout and writes dt_out
# in (batch, nheads, nchunks, chunk_size) layout. This is a TRANSPOSE + RESHAPE.
# Always fp32 for numerical stability.

dt_out_shape = (batch, nheads, nchunks, chunk_size)
print(f"dt_out:    {list(dt_out_shape)}  = (batch, nheads, nchunks, chunk_size)")
print(f"           dtype=float32 always")

# LINE 725: dA_cumsum = torch.empty(batch, nheads, nchunks, chunk_size,
#                                    device=dt.device, dtype=torch.float32)
#
# Allocate output for cumulative dA values. Same shape as dt_out.
# dA_cumsum[b, h, c, i] = sum_{j=0}^{i} (dt_processed[b, c*Q+j, h] * A[h])

dA_cumsum_shape = (batch, nheads, nchunks, chunk_size)
print(f"dA_cumsum: {list(dA_cumsum_shape)}  = (batch, nheads, nchunks, chunk_size)")

dt_out:    [2, 24, 2, 256]  = (batch, nheads, nchunks, chunk_size)
           dtype=float32 always
dA_cumsum: [2, 24, 2, 256]  = (batch, nheads, nchunks, chunk_size)


In [5]:
# LINE 726: grid_chunk_cs = lambda META: (batch, nchunks, triton.cdiv(nheads, META['BLOCK_SIZE_H']))
#
# The GPU grid — how many thread blocks ("programs") to launch.
# Grid is 3D: (batch, nchunks, ceil(nheads / BLOCK_SIZE_H))
#
# Each thread block processes:
#   - One batch element           (axis=0, pid_b)
#   - One chunk                   (axis=1, pid_c)
#   - BLOCK_SIZE_H heads at once  (axis=2, pid_h)
#
# BLOCK_SIZE_H is autotuned from {1, 2, 4, 8, 16, 32, 64}.
# With nheads=24 and BLOCK_SIZE_H=8: grid = (2, 2, 3) = 12 thread blocks.

for bsh in [1, 2, 4, 8, 16, 32, 64]:
    grid = (batch, nchunks, triton.cdiv(nheads, bsh))
    print(f"  BLOCK_SIZE_H={bsh:2d}  =>  grid={grid}  ({grid[0]*grid[1]*grid[2]:3d} blocks, {bsh} heads/block)")

  BLOCK_SIZE_H= 1  =>  grid=(2, 2, 24)  ( 96 blocks, 1 heads/block)
  BLOCK_SIZE_H= 2  =>  grid=(2, 2, 12)  ( 48 blocks, 2 heads/block)
  BLOCK_SIZE_H= 4  =>  grid=(2, 2, 6)  ( 24 blocks, 4 heads/block)
  BLOCK_SIZE_H= 8  =>  grid=(2, 2, 3)  ( 12 blocks, 8 heads/block)
  BLOCK_SIZE_H=16  =>  grid=(2, 2, 2)  (  8 blocks, 16 heads/block)
  BLOCK_SIZE_H=32  =>  grid=(2, 2, 1)  (  4 blocks, 32 heads/block)
  BLOCK_SIZE_H=64  =>  grid=(2, 2, 1)  (  4 blocks, 64 heads/block)


In [6]:
# LINE 728-740: The kernel launch
#
# _chunk_cumsum_fwd_kernel[grid_chunk_cs](
#     dt, A, dt_bias, dt_out, dA_cumsum,       # tensor pointers
#     batch, seqlen, nheads, chunk_size,        # dimensions
#     dt_limit[0], dt_limit[1],                 # clamp bounds (0.0, inf)
#     dt.stride(0), dt.stride(1), dt.stride(2), # dt strides
#     A.stride(0),                              # A stride
#     dt_bias.stride(0) if dt_bias is not None else 0,  # dt_bias stride
#     dt_out.stride(0), dt_out.stride(2), dt_out.stride(1), dt_out.stride(3),  # dt_out strides
#     dA_cumsum.stride(0), dA_cumsum.stride(2), dA_cumsum.stride(1), dA_cumsum.stride(3),
#     dt_softplus,                              # constexpr bool
#     HAS_DT_BIAS=dt_bias is not None,          # constexpr bool
#     BLOCK_SIZE_CHUNK=triton.next_power_of_2(chunk_size),  # constexpr
# )
#
# Let's trace the strides to understand memory layout:

print("=== Input dt strides ===")
print(f"  dt.shape   = {list(dt.shape)}  (batch, seqlen, nheads)")
print(f"  dt.stride  = {dt.stride()}")
print(f"  stride(0) = {dt.stride(0):6d}  # skip {dt.stride(0)} elements to next batch")
print(f"  stride(1) = {dt.stride(1):6d}  # skip {dt.stride(1)} elements to next seq position")
print(f"  stride(2) = {dt.stride(2):6d}  # skip {dt.stride(2)} elements to next head")

# For dt_out and dA_cumsum, the strides passed are NOT in dimension order!
# The code passes: stride(0), stride(2), stride(1), stride(3)
# which maps to:   batch,     chunk,     head,      csize
#
# This is because dt_out shape is (batch, nheads, nchunks, chunk_size)
# but the kernel parameters are named: batch, chunk, head, csize

dt_out_tmp = torch.empty(batch, nheads, nchunks, chunk_size, device=device, dtype=torch.float32)
print(f"\n=== Output dt_out strides ===")
print(f"  dt_out.shape   = {list(dt_out_tmp.shape)}  (batch, nheads, nchunks, chunk_size)")
print(f"  dt_out.stride  = {dt_out_tmp.stride()}")
print(f"  Passed to kernel as: stride(0)={dt_out_tmp.stride(0)}, stride(2)={dt_out_tmp.stride(2)}, "
      f"stride(1)={dt_out_tmp.stride(1)}, stride(3)={dt_out_tmp.stride(3)}")
print(f"  => kernel sees:      stride_batch={dt_out_tmp.stride(0)}, stride_chunk={dt_out_tmp.stride(2)}, "
      f"stride_head={dt_out_tmp.stride(1)}, stride_csize={dt_out_tmp.stride(3)}")

print(f"\n=== BLOCK_SIZE_CHUNK ===")
print(f"  chunk_size = {chunk_size}")
print(f"  next_power_of_2({chunk_size}) = {triton.next_power_of_2(chunk_size)}")
print(f"  (Must be power of 2 for Triton's tl.cumsum to work)")

=== Input dt strides ===
  dt.shape   = [2, 512, 24]  (batch, seqlen, nheads)
  dt.stride  = (12288, 24, 1)
  stride(0) =  12288  # skip 12288 elements to next batch
  stride(1) =     24  # skip 24 elements to next seq position
  stride(2) =      1  # skip 1 elements to next head

=== Output dt_out strides ===
  dt_out.shape   = [2, 24, 2, 256]  (batch, nheads, nchunks, chunk_size)
  dt_out.stride  = (12288, 512, 256, 1)
  Passed to kernel as: stride(0)=12288, stride(2)=256, stride(1)=512, stride(3)=1
  => kernel sees:      stride_batch=12288, stride_chunk=256, stride_head=512, stride_csize=1

=== BLOCK_SIZE_CHUNK ===
  chunk_size = 256
  next_power_of_2(256) = 256
  (Must be power of 2 for Triton's tl.cumsum to work)


---
## Part 2: The Triton kernel

File: `ssd_chunk_state.py:40-86`

Each thread block processes **one (batch, chunk)** pair and **BLOCK_SIZE_H heads**.
Within a single thread block, it loads the entire chunk (up to BLOCK_SIZE_CHUNK positions)
for those heads, processes them, and writes the outputs.

### Autotuning
```python
@triton.autotune(
    configs=[triton.Config({'BLOCK_SIZE_H': 1}),
             triton.Config({'BLOCK_SIZE_H': 2}),
             ...
             triton.Config({'BLOCK_SIZE_H': 64})],
    key=['chunk_size', 'nheads'],   # re-tune if these change
)
```
Triton benchmarks all 7 BLOCK_SIZE_H values and picks the fastest.
The key says: re-run autotuning if `chunk_size` or `nheads` changes.

In [7]:
# ============================================================
# SIMULATING THE KERNEL LINE BY LINE
# ============================================================
# We'll simulate with BLOCK_SIZE_H=8 and BLOCK_SIZE_CHUNK=256
# for one specific thread block: pid_b=0, pid_c=0, pid_h=0

BLOCK_SIZE_H = 8
BLOCK_SIZE_CHUNK = triton.next_power_of_2(chunk_size)  # 256

# Simulate one thread block
pid_b = 0   # which batch
pid_c = 0   # which chunk
pid_h = 0   # which group of heads (0 => heads 0..7)

print(f"Simulating thread block: pid_b={pid_b}, pid_c={pid_c}, pid_h={pid_h}")
print(f"BLOCK_SIZE_H={BLOCK_SIZE_H}, BLOCK_SIZE_CHUNK={BLOCK_SIZE_CHUNK}")
print(f"This block handles: batch {pid_b}, chunk {pid_c}, heads {pid_h*BLOCK_SIZE_H}..{(pid_h+1)*BLOCK_SIZE_H-1}")

Simulating thread block: pid_b=0, pid_c=0, pid_h=0
BLOCK_SIZE_H=8, BLOCK_SIZE_CHUNK=256
This block handles: batch 0, chunk 0, heads 0..7


In [8]:
# ============================================================
# LINE 57-59: Get program IDs (which thread block am I?)
# ============================================================
#   pid_b = tl.program_id(axis=0)    # batch index
#   pid_c = tl.program_id(axis=1)    # chunk index
#   pid_h = tl.program_id(axis=2)    # head-block index
#
# These come from the grid: (batch, nchunks, cdiv(nheads, BLOCK_SIZE_H))
# pid_h=0 handles heads [0, 1, ..., BLOCK_SIZE_H-1]
# pid_h=1 handles heads [BLOCK_SIZE_H, ..., 2*BLOCK_SIZE_H-1]
# etc.

print(f"pid_b={pid_b}  (batch element {pid_b})")
print(f"pid_c={pid_c}  (chunk {pid_c}, covers seq positions {pid_c*chunk_size}..{(pid_c+1)*chunk_size-1})")
print(f"pid_h={pid_h}  (head block {pid_h}, covers heads {pid_h*BLOCK_SIZE_H}..{(pid_h+1)*BLOCK_SIZE_H-1})")

pid_b=0  (batch element 0)
pid_c=0  (chunk 0, covers seq positions 0..255)
pid_h=0  (head block 0, covers heads 0..7)


In [9]:
# ============================================================
# LINE 60-62: Advance base pointers to this thread block's region
# ============================================================
#   dt_ptr += pid_b * stride_dt_batch + pid_c * chunk_size * stride_dt_seqlen
#   dt_out_ptr += pid_b * stride_dt_out_batch + pid_c * stride_dt_out_chunk
#   dA_cumsum_ptr += pid_b * stride_dA_cs_batch + pid_c * stride_dA_cs_chunk
#
# For dt (input): we advance to batch pid_b, then to the start of chunk pid_c.
#   Since dt is (batch, seqlen, nheads), chunk c starts at seqlen position c*chunk_size.
#   So we skip: pid_b * (seqlen*nheads) + pid_c * chunk_size * nheads elements.
#
# For dt_out/dA_cumsum: shape is (batch, nheads, nchunks, chunk_size).
#   We advance to batch pid_b, chunk pid_c.
#   stride_dt_out_chunk is actually dt_out.stride(2) = chunk_size.

# Let's compute the byte offsets
dt_base_offset = pid_b * dt.stride(0) + pid_c * chunk_size * dt.stride(1)
print(f"dt base offset: {dt_base_offset} elements")
print(f"  = pid_b({pid_b}) * stride_batch({dt.stride(0)}) + pid_c({pid_c}) * chunk_size({chunk_size}) * stride_seqlen({dt.stride(1)})")
print(f"  This points to dt[{pid_b}, {pid_c*chunk_size}, 0]")

dt base offset: 0 elements
  = pid_b(0) * stride_batch(12288) + pid_c(0) * chunk_size(256) * stride_seqlen(24)
  This points to dt[0, 0, 0]


In [10]:
# ============================================================
# LINE 64-65: Create offset vectors for heads and chunk positions
# ============================================================
#   offs_h = pid_h * BLOCK_SIZE_H + tl.arange(0, BLOCK_SIZE_H)
#   offs_c = tl.arange(0, BLOCK_SIZE_CHUNK)
#
# offs_h: which heads this block handles. Shape: (BLOCK_SIZE_H,)
# offs_c: positions within the chunk.       Shape: (BLOCK_SIZE_CHUNK,)

offs_h = pid_h * BLOCK_SIZE_H + torch.arange(BLOCK_SIZE_H)  # e.g. [0, 1, 2, ..., 7]
offs_c = torch.arange(BLOCK_SIZE_CHUNK)                       # [0, 1, 2, ..., 255]

print(f"offs_h: {offs_h.tolist()}  (which heads to process)")
print(f"offs_c: [0, 1, ..., {BLOCK_SIZE_CHUNK-1}]  (positions within chunk, {BLOCK_SIZE_CHUNK} total)")

offs_h: [0, 1, 2, 3, 4, 5, 6, 7]  (which heads to process)
offs_c: [0, 1, ..., 255]  (positions within chunk, 256 total)


In [11]:
# ============================================================
# LINE 66: Build 2D pointer grid for loading dt
# ============================================================
#   dt_ptrs = dt_ptr + (offs_h[:, None] * stride_dt_head + offs_c[None, :] * stride_dt_seqlen)
#
# This creates a 2D array of pointers: (BLOCK_SIZE_H, BLOCK_SIZE_CHUNK)
# Each entry points to dt[pid_b, pid_c*chunk_size + offs_c, offs_h]
#
# The 2D structure:
#   Row axis = heads (BLOCK_SIZE_H)
#   Col axis = positions within chunk (BLOCK_SIZE_CHUNK)
#
# offs_h[:, None] broadcasts to (BLOCK_SIZE_H, 1)
# offs_c[None, :] broadcasts to (1, BLOCK_SIZE_CHUNK)
# Result: (BLOCK_SIZE_H, BLOCK_SIZE_CHUNK) offset grid

print(f"dt_ptrs shape: ({BLOCK_SIZE_H}, {BLOCK_SIZE_CHUNK})")
print(f"")
print(f"  dt_ptrs[h, c] points to: dt[batch={pid_b}, seq={pid_c*chunk_size}+c, head=offs_h[h]]")
print(f"")
print(f"  Conceptually:")
print(f"                 pos 0    pos 1    pos 2   ...  pos {chunk_size-1}")
print(f"    head {offs_h[0].item()}:    dt[0,0,{offs_h[0].item()}] dt[0,1,{offs_h[0].item()}] dt[0,2,{offs_h[0].item()}]  ...")
print(f"    head {offs_h[1].item()}:    dt[0,0,{offs_h[1].item()}] dt[0,1,{offs_h[1].item()}] dt[0,2,{offs_h[1].item()}]  ...")
print(f"    ...")
print(f"    head {offs_h[-1].item()}:    dt[0,0,{offs_h[-1].item()}] dt[0,1,{offs_h[-1].item()}] dt[0,2,{offs_h[-1].item()}]  ...")

dt_ptrs shape: (8, 256)

  dt_ptrs[h, c] points to: dt[batch=0, seq=0+c, head=offs_h[h]]

  Conceptually:
                 pos 0    pos 1    pos 2   ...  pos 255
    head 0:    dt[0,0,0] dt[0,1,0] dt[0,2,0]  ...
    head 1:    dt[0,0,1] dt[0,1,1] dt[0,2,1]  ...
    ...
    head 7:    dt[0,0,7] dt[0,1,7] dt[0,2,7]  ...


In [12]:
# ============================================================
# LINE 67: Build pointer for A
# ============================================================
#   A_ptrs = A_ptr + offs_h * stride_A_head
#
# A is (nheads,), so A_ptrs is just (BLOCK_SIZE_H,)
# Each entry points to A[offs_h[i]]

print(f"A_ptrs: ({BLOCK_SIZE_H},)")
print(f"  Points to A[{offs_h[0].item()}], A[{offs_h[1].item()}], ..., A[{offs_h[-1].item()}]")
print(f"  Values: {A[offs_h[0]:offs_h[-1]+1]}")

A_ptrs: (8,)
  Points to A[0], A[1], ..., A[7]
  Values: tensor([-0.9877, -0.1289, -0.5621, -0.5221, -0.7445, -0.5955, -0.9647, -0.8979],
       device='cuda:0')


In [13]:
# ============================================================
# LINE 68-69: Build 2D pointer grids for outputs (same structure as dt_ptrs)
# ============================================================
#   dt_out_ptrs = dt_out_ptr + (offs_h[:, None] * stride_dt_out_head
#                              + offs_c[None, :] * stride_dt_out_csize)
#   dA_cs_ptrs = dA_cumsum_ptr + (offs_h[:, None] * stride_dA_cs_head
#                                + offs_c[None, :] * stride_dA_cs_csize)
#
# Same 2D structure: (BLOCK_SIZE_H, BLOCK_SIZE_CHUNK)
# dt_out_ptrs[h, c] -> dt_out[pid_b, offs_h[h], pid_c, c]
# Note: the output layout is (batch, nheads, nchunks, chunk_size)
#   so the kernel is doing an implicit transpose from
#   input  (batch, seqlen, nheads)  to
#   output (batch, nheads, nchunks, chunk_size)

print(f"dt_out_ptrs and dA_cs_ptrs: both ({BLOCK_SIZE_H}, {BLOCK_SIZE_CHUNK})")
print(f"  dt_out_ptrs[h, c] -> dt_out[batch={pid_b}, head=offs_h[h], chunk={pid_c}, pos=c]")

dt_out_ptrs and dA_cs_ptrs: both (8, 256)
  dt_out_ptrs[h, c] -> dt_out[batch=0, head=offs_h[h], chunk=0, pos=c]


In [14]:
# ============================================================
# LINE 70: Compute chunk_size_limit
# ============================================================
#   chunk_size_limit = min(chunk_size, seqlen - pid_c * chunk_size)
#
# This handles the LAST chunk which might be shorter than chunk_size.
# Example: seqlen=500, chunk_size=256
#   chunk 0: chunk_size_limit = min(256, 500 - 0) = 256  (full chunk)
#   chunk 1: chunk_size_limit = min(256, 500 - 256) = 244  (partial chunk!)
#
# In our case seqlen=512, chunk_size=256, so both chunks are full.

for c in range(nchunks):
    limit = min(chunk_size, seqlen - c * chunk_size)
    print(f"  chunk {c}: chunk_size_limit = min({chunk_size}, {seqlen} - {c*chunk_size}) = {limit}"
          f"{'  (partial!)' if limit < chunk_size else ''}")

  chunk 0: chunk_size_limit = min(256, 512 - 0) = 256
  chunk 1: chunk_size_limit = min(256, 512 - 256) = 256


In [15]:
# ============================================================
# LINE 72: Load dt values into registers
# ============================================================
#   dt = tl.load(dt_ptrs,
#                mask=(offs_h[:, None] < nheads) & (offs_c[None, :] < chunk_size_limit),
#                other=0.0).to(tl.float32)
#
# Loads a (BLOCK_SIZE_H, BLOCK_SIZE_CHUNK) tile of dt values.
# The mask handles two edge cases:
#   1. offs_h < nheads:        last head block might go past nheads
#      e.g., nheads=24, BLOCK_SIZE_H=8: block 2 has offs_h=[16..23], all valid
#      but if BLOCK_SIZE_H=32: block 0 has offs_h=[0..31], but only 0..23 valid!
#   2. offs_c < chunk_size_limit: last chunk might be partial
#
# Out-of-bounds values are set to 0.0.
# Result is cast to float32 for numerical stability.

# Simulate the load
chunk_size_limit = min(chunk_size, seqlen - pid_c * chunk_size)
mask_h = offs_h < nheads                                      # (BLOCK_SIZE_H,)
mask_c = offs_c < chunk_size_limit                             # (BLOCK_SIZE_CHUNK,)
mask_2d = mask_h[:, None] & mask_c[None, :]                    # (BLOCK_SIZE_H, BLOCK_SIZE_CHUNK)

# Load: dt[pid_b, pid_c*chunk_size + offs_c, offs_h]
dt_loaded = torch.zeros(BLOCK_SIZE_H, BLOCK_SIZE_CHUNK, device=device, dtype=torch.float32)
for hi in range(BLOCK_SIZE_H):
    for ci in range(BLOCK_SIZE_CHUNK):
        if mask_2d[hi, ci]:
            dt_loaded[hi, ci] = dt[pid_b, pid_c * chunk_size + ci, offs_h[hi]]

print(f"dt_loaded shape: ({BLOCK_SIZE_H}, {BLOCK_SIZE_CHUNK})")
print(f"dt_loaded[0, :8] (head {offs_h[0].item()}, first 8 positions): {dt_loaded[0, :8]}")
print(f"\nVerify: dt[0, 0, 0] = {dt[pid_b, pid_c*chunk_size, offs_h[0]].item():.4f}")
print(f"        dt_loaded[0,0] = {dt_loaded[0, 0].item():.4f}")

dt_loaded shape: (8, 256)
dt_loaded[0, :8] (head 0, first 8 positions): tensor([ 0.1940, -0.3260, -0.3347,  0.3617,  0.5960,  0.7486, -0.6326, -1.0732],
       device='cuda:0')

Verify: dt[0, 0, 0] = 0.1940
        dt_loaded[0,0] = 0.1940


In [16]:
# ============================================================
# LINE 73-75: Add dt_bias (if present)
# ============================================================
#   if HAS_DT_BIAS:
#       dt_bias = tl.load(dt_bias_ptr + offs_h * stride_dt_bias_head,
#                         mask=offs_h < nheads, other=0.0).to(tl.float32)
#       dt += dt_bias[:, None]
#
# Load dt_bias for each head: shape (BLOCK_SIZE_H,)
# Then broadcast-add to all positions: dt_bias[:, None] is (BLOCK_SIZE_H, 1)
# So each head gets the SAME bias added to ALL positions.

dt_bias_loaded = dt_bias[offs_h[0]:offs_h[-1]+1].float()  # (BLOCK_SIZE_H,)
print(f"dt_bias_loaded: {dt_bias_loaded[:4]}  (first 4 heads)")

dt_after_bias = dt_loaded + dt_bias_loaded[:, None]  # (BLOCK_SIZE_H, BLOCK_SIZE_CHUNK)
print(f"\nBefore bias: dt_loaded[0, 0] = {dt_loaded[0, 0]:.4f}")
print(f"Bias[head 0]:                  = {dt_bias_loaded[0]:.4f}")
print(f"After bias:  dt_after_bias[0,0] = {dt_after_bias[0, 0]:.4f}")

dt_bias_loaded: tensor([-0.5187,  1.2268,  0.6255, -0.9117], device='cuda:0')  (first 4 heads)

Before bias: dt_loaded[0, 0] = 0.1940
Bias[head 0]:                  = -0.5187
After bias:  dt_after_bias[0,0] = -0.3247


In [17]:
# ============================================================
# LINE 76-77: Apply softplus (if enabled)
# ============================================================
#   if DT_SOFTPLUS:
#       dt = tl.where(dt <= 20.0, softplus(dt), dt)
#
# softplus(x) = log(1 + exp(x))
#
# The tl.where guard: for dt > 20, softplus(dt) ≈ dt (because exp(20) >> 1),
# so we skip the computation to avoid exp overflow.
#
# Purpose: softplus ensures dt is ALWAYS POSITIVE.
# This is critical because dt * A must be negative (A < 0),
# so dt must be positive.

dt_after_softplus = torch.where(
    dt_after_bias <= 20.0,
    F.softplus(dt_after_bias),
    dt_after_bias,
)

print(f"Before softplus: dt_after_bias[0, :4]    = {dt_after_bias[0, :4]}")
print(f"After softplus:  dt_after_softplus[0, :4] = {dt_after_softplus[0, :4]}")
print(f"\nSoftplus maps:")
print(f"  negative -> small positive  (e.g. softplus(-2) = {F.softplus(torch.tensor(-2.0)):.4f})")
print(f"  zero -> ln(2)               (softplus(0)  = {F.softplus(torch.tensor(0.0)):.4f})")
print(f"  positive -> ~itself          (softplus(5)  = {F.softplus(torch.tensor(5.0)):.4f})")

Before softplus: dt_after_bias[0, :4]    = tensor([-0.3247, -0.8448, -0.8534, -0.1571], device='cuda:0')
After softplus:  dt_after_softplus[0, :4] = tensor([0.5439, 0.3574, 0.3548, 0.6177], device='cuda:0')

Softplus maps:
  negative -> small positive  (e.g. softplus(-2) = 0.1269)
  zero -> ln(2)               (softplus(0)  = 0.6931)
  positive -> ~itself          (softplus(5)  = 5.0067)


In [18]:
# ============================================================
# LINE 78-81: Clamp dt to [dt_min, dt_max] and zero out-of-bounds
# ============================================================
#   dt = tl.minimum(tl.maximum(dt, dt_min), dt_max)
#   dt = tl.where((offs_h[:, None] < nheads) & (offs_c[None, :] < chunk_size_limit), dt, 0.0)
#
# First: clamp to [dt_min=0.0, dt_max=inf] (defaults). With softplus already applied,
# dt is already positive, so the clamp with default limits is a no-op.
# But if the user sets dt_limit=(0.01, 1.0), this enforces those bounds.
#
# Second: zero out any values that fell outside the valid mask.
# This ensures padding positions in the last chunk don't contribute to the cumsum.

dt_min, dt_max = 0.0, float("inf")
dt_clamped = torch.clamp(dt_after_softplus, min=dt_min, max=dt_max)

# Zero out invalid positions
dt_clamped = torch.where(mask_2d.to(device), dt_clamped, torch.zeros_like(dt_clamped))

print(f"After clamp+mask: dt_clamped[0, :8] = {dt_clamped[0, :8]}")
print(f"All positive: {(dt_clamped >= 0).all()}")

After clamp+mask: dt_clamped[0, :8] = tensor([0.5439, 0.3574, 0.3548, 0.6177, 0.7325, 0.8146, 0.2748, 0.1853],
       device='cuda:0')
All positive: True


In [19]:
# ============================================================
# LINE 82: Store processed dt to dt_out
# ============================================================
#   tl.store(dt_out_ptrs, dt,
#            mask=(offs_h[:, None] < nheads) & (offs_c[None, :] < chunk_size))
#
# Write the processed dt values to dt_out.
# Note the mask uses chunk_size (not chunk_size_limit) — this is because
# dt_out was allocated with full chunk_size, even if the last chunk is partial.
# The extra positions are already 0 from the tl.where above.

print(f"Storing dt_clamped to dt_out[{pid_b}, heads {offs_h[0].item()}..{offs_h[-1].item()}, chunk {pid_c}, :]")
print(f"Shape written: ({BLOCK_SIZE_H}, {BLOCK_SIZE_CHUNK})")

Storing dt_clamped to dt_out[0, heads 0..7, chunk 0, :]
Shape written: (8, 256)


In [20]:
# ============================================================
# LINE 83: Load A values
# ============================================================
#   A = tl.load(A_ptrs, mask=offs_h < nheads, other=0.0).to(tl.float32)
#
# Load A for the heads in this block. Shape: (BLOCK_SIZE_H,)
# A is constant per head — same value for all positions in the chunk.

A_loaded = A[offs_h[0]:offs_h[-1]+1].float()  # (BLOCK_SIZE_H,)
print(f"A_loaded: {A_loaded}")
print(f"All negative: {(A_loaded < 0).all()}")

A_loaded: tensor([-0.9877, -0.1289, -0.5621, -0.5221, -0.7445, -0.5955, -0.9647, -0.8979],
       device='cuda:0')
All negative: True


In [21]:
# ============================================================
# LINE 84: Compute dA = dt * A
# ============================================================
#   dA = dt * A[:, None]
#
# A[:, None] broadcasts (BLOCK_SIZE_H,) -> (BLOCK_SIZE_H, 1)
# dt is (BLOCK_SIZE_H, BLOCK_SIZE_CHUNK)
# Result: dA is (BLOCK_SIZE_H, BLOCK_SIZE_CHUNK)
#
# dA[h, i] = dt_processed[h, i] * A[h]
# Since dt > 0 and A < 0, dA is ALWAYS NEGATIVE.
# This is the per-step log-decay: exp(dA) is the decay factor in (0, 1).

dA = dt_clamped * A_loaded[:, None]  # (BLOCK_SIZE_H, BLOCK_SIZE_CHUNK)

print(f"dA shape: ({BLOCK_SIZE_H}, {BLOCK_SIZE_CHUNK})")
print(f"dA[0, :8] = {dA[0, :8]}")
print(f"All negative: {(dA[dA != 0] < 0).all()}")
print(f"\nInterpretation: dA[h, i] is the log-decay at position i for head h")
print(f"  exp(dA[0, 0]) = {torch.exp(dA[0, 0]):.4f}  (state retention at position 0, head 0)")

dA shape: (8, 256)
dA[0, :8] = tensor([-0.5372, -0.3530, -0.3505, -0.6101, -0.7235, -0.8046, -0.2714, -0.1830],
       device='cuda:0')
All negative: True

Interpretation: dA[h, i] is the log-decay at position i for head h
  exp(dA[0, 0]) = 0.5844  (state retention at position 0, head 0)


In [22]:
# ============================================================
# LINE 85: Compute cumulative sum of dA within the chunk
# ============================================================
#   dA_cs = tl.cumsum(dA, axis=1)
#
# Cumulative sum along the chunk position axis (axis=1).
# dA_cs[h, i] = dA[h, 0] + dA[h, 1] + ... + dA[h, i]
#             = sum_{j=0}^{i} dt[j] * A[h]
#
# This is the total log-decay from the start of the chunk to position i.
# Since all dA values are negative, dA_cs is monotonically DECREASING.
#
# KEY PROPERTY: exp(dA_cs[i] - dA_cs[j]) = product of per-step decays from j+1 to i
#             = the total decay from position j to position i
#
# NOTE: tl.cumsum requires axis dimension to be a power of 2,
#       which is why BLOCK_SIZE_CHUNK = next_power_of_2(chunk_size).

dA_cs = torch.cumsum(dA, dim=1)  # (BLOCK_SIZE_H, BLOCK_SIZE_CHUNK)

print(f"dA_cs shape: ({BLOCK_SIZE_H}, {BLOCK_SIZE_CHUNK})")
print(f"dA_cs[0, :8] = {dA_cs[0, :8]}")
print(f"Monotonically decreasing: {(dA_cs[0, 1:chunk_size_limit] <= dA_cs[0, :chunk_size_limit-1] + 1e-6).all()}")
print(f"\nTotal decay over chunk (head 0): exp(dA_cs[0, {chunk_size-1}]) = {torch.exp(dA_cs[0, chunk_size-1]):.6e}")
print(f"dA_cs[0, {chunk_size-1}] = {dA_cs[0, chunk_size-1]:.2f}  (very negative => almost complete decay)")

dA_cs shape: (8, 256)
dA_cs[0, :8] = tensor([-0.5372, -0.8903, -1.2407, -1.8509, -2.5744, -3.3790, -3.6504, -3.8334],
       device='cuda:0')
Monotonically decreasing: True

Total decay over chunk (head 0): exp(dA_cs[0, 255]) = 0.000000e+00
dA_cs[0, 255] = -154.47  (very negative => almost complete decay)


In [23]:
# ============================================================
# LINE 86: Store dA_cumsum
# ============================================================
#   tl.store(dA_cs_ptrs, dA_cs,
#            mask=(offs_h[:, None] < nheads) & (offs_c[None, :] < chunk_size))
#
# Write cumulative decay values to dA_cumsum output tensor.
# Same mask as dt_out: full chunk_size, not chunk_size_limit.

print(f"Storing dA_cs to dA_cumsum[{pid_b}, heads {offs_h[0].item()}..{offs_h[-1].item()}, chunk {pid_c}, :]")
print(f"Shape written: ({BLOCK_SIZE_H}, {BLOCK_SIZE_CHUNK})")

Storing dA_cs to dA_cumsum[0, heads 0..7, chunk 0, :]
Shape written: (8, 256)


---
## Part 3: End-to-end verification

Let's verify our manual simulation matches the actual kernel output.

In [24]:
# Run the actual kernel
dA_cumsum_actual, dt_out_actual = _chunk_cumsum_fwd(
    dt, A, chunk_size,
    dt_bias=dt_bias,
    dt_softplus=True,
    dt_limit=(0.0, float("inf")),
)

# Compare our manual simulation (for pid_b=0, pid_c=0, heads 0..7)
# dt_out_actual is (batch, nheads, nchunks, chunk_size)
dt_out_slice = dt_out_actual[pid_b, offs_h[0]:offs_h[-1]+1, pid_c, :BLOCK_SIZE_CHUNK]
dA_cs_slice = dA_cumsum_actual[pid_b, offs_h[0]:offs_h[-1]+1, pid_c, :BLOCK_SIZE_CHUNK]

dt_diff = (dt_clamped - dt_out_slice).abs().max().item()
dA_diff = (dA_cs - dA_cs_slice).abs().max().item()

print(f"dt_out  manual vs kernel max diff: {dt_diff:.2e}")
print(f"dA_cs   manual vs kernel max diff: {dA_diff:.2e}")
print(f"\ndt_out  match: {dt_diff < 1e-5}")
print(f"dA_cs   match: {dA_diff < 1e-4}")

dt_out  manual vs kernel max diff: 2.38e-07
dA_cs   manual vs kernel max diff: 3.05e-05

dt_out  match: True
dA_cs   match: True


---
## Summary: Complete data flow

```
dt (B, L, H)           Raw time-step values from in_proj
     |                   (can be any real number)
     v
 + dt_bias (H,)         Add per-head bias (learned parameter)
     |                   dt[b,t,h] += dt_bias[h]
     v
  softplus               Ensure positivity: log(1 + exp(x))
     |                   (skipped if x > 20 for numerical safety)
     v
  clamp [dt_min, dt_max] Enforce bounds (default [0, inf] = no-op)
     |                   
     v
 dt_out (B, H, K, Q)    Processed dt, stored as output
     |                   (note the transpose: L -> K,Q and H moves to dim 1)
     v
  * A (H,)               Multiply by per-head decay rate (negative)
     |                   dA[b,h,c,i] = dt_out[b,h,c,i] * A[h]
     v
  cumsum (per chunk)     Cumulative sum along positions within each chunk
     |                   dA_cs[b,h,c,i] = sum_{j=0}^{i} dA[b,h,c,j]
     v
 dA_cumsum (B, H, K, Q) Total log-decay from chunk start to each position
```

### GPU execution model
```
Grid: (batch, nchunks, ceil(nheads / BLOCK_SIZE_H))

Each thread block:
  - Handles 1 batch, 1 chunk, BLOCK_SIZE_H heads
  - Loads (BLOCK_SIZE_H, BLOCK_SIZE_CHUNK) tile of dt
  - Processes all positions in parallel (per-element ops + cumsum)
  - Writes (BLOCK_SIZE_H, BLOCK_SIZE_CHUNK) tile of dt_out and dA_cumsum
  - Total work per block: O(BLOCK_SIZE_H * BLOCK_SIZE_CHUNK)
```